# 00 · Inventario de conversaciones Mantra
Descubre automáticamente los archivos de `MyDrive/mine_chatbot`, formatos, tamaños, columnas y duplicados. No modifica los datos originales.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!git clone -q https://github.com/dinatalediego/ai_assistant.git || (cd ai_assistant && git pull)
%cd /content/ai_assistant
!pip -q install pandas openpyxl pyarrow pyyaml

In [ ]:
from pathlib import Path
from src.drive_inventory import build_inventory
DRIVE_ROOT = Path('/content/drive/MyDrive/mine_chatbot')
assert DRIVE_ROOT.exists(), f'No existe: {DRIVE_ROOT}'
inventory = build_inventory(DRIVE_ROOT)
print('Archivos soportados:', len(inventory))
display(inventory.head(20))

In [ ]:
display(inventory.groupby('suffix', dropna=False).agg(archivos=('name','size'), bytes=('bytes','sum')).sort_values('archivos', ascending=False))
duplicates = inventory[inventory.duplicated('sha256', keep=False)].sort_values('sha256')
print('Archivos que participan en duplicados exactos:', len(duplicates))
display(duplicates[['name','path','bytes','sha256']].head(50))

In [ ]:
from collections import Counter
column_sets = Counter(tuple(x) for x in inventory['columns'].dropna())
for cols, n in column_sets.most_common(10):
    print(f'\n{n} archivo(s) · {len(cols)} columnas')
    print(cols)

In [ ]:
OUT = Path('/content/drive/MyDrive/mine_chatbot/_analysis_outputs')
OUT.mkdir(exist_ok=True)
inventory.to_csv(OUT/'file_inventory.csv', index=False)
duplicates.to_csv(OUT/'exact_duplicates.csv', index=False)
print('Resultados guardados en:', OUT)